# Week 3: Model Training & Selection
## Spam Classifier Project

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

## Step 1: Load Data

In [ ]:
# Load cleaned data
df = pd.read_csv('../data/cleaned_data.csv')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

## Step 2: TF-IDF Vectorization

In [ ]:
# TF-IDF vectorization
tfidf_vectorizer = TfidfVectorizer(max_features=3000, stop_words='english')
X_tfidf = tfidf_vectorizer.fit_transform(df['cleaned_text'])
y = df['target']

print(f"TF-IDF matrix shape: {X_tfidf.shape}")

## Step 3: Train Test Split

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set: {X_train.shape[0]}")
print(f"Test set: {X_test.shape[0]}")

## Step 4: Train Models

In [ ]:
# Train models
models = {}
predictions = {}

# Gaussian NB
models['Gaussian NB'] = GaussianNB()
models['Gaussian NB'].fit(X_train.toarray(), y_train)
predictions['Gaussian NB'] = models['Gaussian NB'].predict(X_test.toarray())

# Multinomial NB
models['Multinomial NB'] = MultinomialNB()
models['Multinomial NB'].fit(X_train, y_train)
predictions['Multinomial NB'] = models['Multinomial NB'].predict(X_test)

# Bernoulli NB
models['Bernoulli NB'] = BernoulliNB()
models['Bernoulli NB'].fit(X_train, y_train)
predictions['Bernoulli NB'] = models['Bernoulli NB'].predict(X_test)

# SVM
models['SVM'] = SVC(kernel='rbf', random_state=42)
models['SVM'].fit(X_train, y_train)
predictions['SVM'] = models['SVM'].predict(X_test)

# Random Forest
models['Random Forest'] = RandomForestClassifier(n_estimators=100, random_state=42)
models['Random Forest'].fit(X_train.toarray(), y_train)
predictions['Random Forest'] = models['Random Forest'].predict(X_test.toarray())

print("✓ All models trained")

## Step 5: Evaluate Models

In [ ]:
# Evaluate
results = []

for model_name, y_pred in predictions.items():
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    results.append({
        'Model': model_name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    })
    
    print(f"\n{model_name}:")
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")

results_df = pd.DataFrame(results)
print("\n" + "="*80)
print(results_df.to_string(index=False))

## Step 6: Confusion Matrix

In [ ]:
# Confusion matrix for best model
best_pred = predictions['Multinomial NB']
cm = confusion_matrix(y_test, best_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Ham', 'Spam'],
            yticklabels=['Ham', 'Spam'])
plt.title('Confusion Matrix - Multinomial Naive Bayes')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('../visualizations/06_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.close()
print("✓ 06_confusion_matrix.png saved")

## Step 7: Model Comparison Chart

In [ ]:
# Model comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(results_df['Model'], results_df['Accuracy'], color=['#3498db', '#2ecc71', '#f39c12', '#e74c3c', '#9b59b6'])
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Model Accuracy Comparison')
axes[0].set_ylim([0.9, 1.0])
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(results_df['Model'], results_df['Precision'], color=['#3498db', '#2ecc71', '#f39c12', '#e74c3c', '#9b59b6'])
axes[1].set_ylabel('Precision')
axes[1].set_title('Model Precision Comparison')
axes[1].set_ylim([0.9, 1.05])
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../visualizations/07_model_comparison.png', dpi=300, bbox_inches='tight')
plt.close()
print("✓ 07_model_comparison.png saved")

## Step 8: Export Models

In [ ]:
# Create models directory
os.makedirs('../models', exist_ok=True)

# Save vectorizer
with open('../models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)

# Save model
with open('../models/spam_classifier_model.pkl', 'wb') as f:
    pickle.dump(models['Multinomial NB'], f)

print("✓ Models exported")
print("  - tfidf_vectorizer.pkl")
print("  - spam_classifier_model.pkl")